In [ ]:
# ============================================================================
# CUSTOMER SEGMENTATION USING RFM ANALYSIS AND KMEANS CLUSTERING
# ============================================================================
# Deliverable: Senior Marketing Analyst -> Retail Executive Team
# Purpose: Segment customers by purchasing behavior (Recency, Frequency,
#          Monetary value) to guide retention strategy and marketing spend.
# Environment: Google Colab (single-cell, end-to-end execution)
# ============================================================================

# ----------------------------------------------------------------------------
# SECTION 0: IMPORTS
# ----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("=" * 90)
print("CUSTOMER SEGMENTATION PIPELINE — RFM ANALYSIS + KMEANS CLUSTERING")
print("=" * 90)

# ============================================================================
# SECTION 1: DATA LOADING AND VALIDATION
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 1: DATA LOADING AND VALIDATION")
print("=" * 90)

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

print(f"\nLoading dataset from UCI ML Repository:\n  {DATA_URL}")
try:
    df_raw = pd.read_excel(DATA_URL, engine="openpyxl")
    print(f"Successfully loaded dataset. Shape: {df_raw.shape}")
except Exception as e:
    raise RuntimeError(
        f"Failed to load dataset from UCI repository. Check network access "
        f"or mirror availability. Original error: {e}"
    )

total_rows_loaded = len(df_raw)

# --- Schema report -----------------------------------------------------
print("\n--- Dataset Schema (Column Names and Data Types) ---")
schema_df = pd.DataFrame({
    "Column": df_raw.columns,
    "Dtype": [str(t) for t in df_raw.dtypes]
})
print(schema_df.to_string(index=False))

# --- Missing value / duplicate diagnostics -----------------------------
print("\n--- Missing Value Report (raw data) ---")
missing_report = df_raw.isnull().sum()
missing_pct = (missing_report / total_rows_loaded * 100).round(2)
missing_summary = pd.DataFrame({
    "Missing_Count": missing_report,
    "Missing_Pct": missing_pct
})
print(missing_summary[missing_summary["Missing_Count"] > 0].to_string())

n_exact_duplicates = df_raw.duplicated().sum()
print(f"\nExact duplicate rows found: {n_exact_duplicates}")

# --- Step-by-step cleaning with row-count tracking ----------------------
df = df_raw.copy()

# Drop exact duplicate transaction rows (data entry duplicates).
df = df.drop_duplicates()
rows_after_dedup = len(df)
rows_removed_duplicates = total_rows_loaded - rows_after_dedup

# Drop rows without a CustomerID — RFM analysis requires a customer key;
# anonymous transactions cannot be attributed to any individual and would
# corrupt per-customer aggregates.
rows_before_custid = len(df)
df = df.dropna(subset=["CustomerID"])
rows_after_custid = len(df)
rows_removed_missing_custid = rows_before_custid - rows_after_custid

# Remove cancelled orders — invoice numbers starting with "C" represent
# returns/cancellations. Including them would double-count or negate
# genuine purchase behavior in the Monetary and Frequency metrics.
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
rows_before_cancel = len(df)
df = df[~df["InvoiceNo"].str.startswith("C")]
rows_after_cancel = len(df)
rows_removed_cancellations = rows_before_cancel - rows_after_cancel

# Remove non-positive quantities — these typically indicate returns,
# stock adjustments, or data entry errors rather than genuine sales.
rows_before_qty = len(df)
df = df[df["Quantity"] > 0]
rows_after_qty = len(df)
rows_removed_bad_quantity = rows_before_qty - rows_after_qty

rows_remaining = len(df)
pct_retained = round(rows_remaining / total_rows_loaded * 100, 2)

print("\n--- DATA QUALITY REPORT ---")
print(f"Total rows loaded:                     {total_rows_loaded:,}")
print(f"Rows removed (exact duplicates):        {rows_removed_duplicates:,}")
print(f"Rows removed (missing CustomerID):      {rows_removed_missing_custid:,}")
print(f"Rows removed (cancelled invoices 'C'):   {rows_removed_cancellations:,}")
print(f"Rows removed (zero/negative quantity):   {rows_removed_bad_quantity:,}")
print(f"Rows remaining after cleaning:           {rows_remaining:,}")
print(f"Percentage of original data retained:    {pct_retained}%")

# ============================================================================
# SECTION 2: DATA PREPROCESSING AND FEATURE ENGINEERING
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 2: DATA PREPROCESSING AND FEATURE ENGINEERING")
print("=" * 90)

# Convert InvoiceDate to datetime, coercing unparseable values to NaT
# rather than throwing, so a handful of malformed timestamps don't halt
# the entire pipeline.
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
n_bad_dates = df["InvoiceDate"].isna().sum()
if n_bad_dates > 0:
    print(f"Dropping {n_bad_dates} rows with unparseable InvoiceDate values.")
    df = df.dropna(subset=["InvoiceDate"])

# Feature engineering: monetary value of each line item.
df["Total_Transaction_Value"] = df["Quantity"] * df["UnitPrice"]

# Remove non-positive transaction values (e.g., free samples, price
# errors) — these don't represent genuine revenue-generating purchases.
rows_before_value_filter = len(df)
df = df[df["Total_Transaction_Value"] > 0]
rows_removed_bad_value = rows_before_value_filter - len(df)
print(f"Rows removed (zero/negative Total_Transaction_Value): {rows_removed_bad_value:,}")
print(f"Final cleaned row count: {len(df):,}")

# --- Descriptive statistics on cleaned data -----------------------------
print("\n--- Descriptive Statistics (Post-Cleaning) ---")
desc_cols = ["Quantity", "UnitPrice", "Total_Transaction_Value"]
desc_stats = df[desc_cols].agg(["count", "mean", "median", "min", "max", "std"]).T
desc_stats.columns = ["Count", "Mean", "Median", "Min", "Max", "Std_Dev"]
print(desc_stats.round(2).to_string())

# --- Date range -----------------------------------------------------------
min_date = df["InvoiceDate"].min()
max_date = df["InvoiceDate"].max()
print(f"\nTransaction date range: {min_date.date()}  to  {max_date.date()}")

# --- Cardinality summary ---------------------------------------------------
n_unique_customers = df["CustomerID"].nunique()
n_unique_products = df["StockCode"].nunique()
n_unique_countries = df["Country"].nunique()
print(f"Unique customers: {n_unique_customers:,}")
print(f"Unique products (StockCode): {n_unique_products:,}")
print(f"Unique countries: {n_unique_countries:,}")

# ============================================================================
# SECTION 3: RFM METRICS CALCULATION
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 3: RFM METRICS CALCULATION")
print("=" * 90)

# Reference date = one day after the last transaction in the dataset.
# This ensures every customer has Recency >= 1 (avoids a Recency of 0,
# which would be an edge case for downstream scaling/interpretation).
reference_date = max_date + pd.Timedelta(days=1)
print(f"Reference date for Recency calculation: {reference_date.date()}")

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Total_Transaction_Value", "sum")
).reset_index()

print(f"\nRFM table constructed for {len(rfm):,} unique customers.")

# --- Statistical summary of RFM metrics ------------------------------------
print("\n--- RFM Statistical Summary ---")
rfm_summary = rfm[["Recency", "Frequency", "Monetary"]].agg(
    ["count", "mean", "median", "min", "max", "std",
     lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]
)
rfm_summary.index = ["Count", "Mean", "Median", "Min", "Max", "Std_Dev", "P25", "P75"]
print(rfm_summary.round(2).to_string())

# --- Top 10 customers by Monetary value ------------------------------------
print("\n--- Top 10 Customers by Monetary Value ---")
top10 = rfm.sort_values("Monetary", ascending=False).head(10)
print(top10[["CustomerID", "Recency", "Frequency", "Monetary"]]
      .round(2).to_string(index=False))

# ============================================================================
# SECTION 4: CUSTOMER SEGMENTATION USING KMEANS CLUSTERING
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 4: CUSTOMER SEGMENTATION USING KMEANS CLUSTERING")
print("=" * 90)

# --- Scaling rationale ------------------------------------------------------
print("""
Why scaling is necessary:
KMeans relies on Euclidean distance to assign points to clusters. Recency
is measured in days (range: single digits to hundreds), Frequency is a
small integer count, and Monetary can range from single dollars to tens
of thousands. Without scaling, Monetary would dominate the distance
calculation purely due to its larger numeric magnitude, effectively
drowning out the signal from Recency and Frequency. StandardScaler
transforms each feature to zero mean and unit variance, ensuring all
three RFM dimensions contribute proportionally to cluster formation.
""")

X = rfm[["Recency", "Frequency", "Monetary"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Elbow method ------------------------------------------------------------
print("Computing inertia for k = 2 to 10 (Elbow Method)...")
k_range = range(2, 11)
inertias = []
for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    print(f"  k={k:2d}  ->  inertia = {km.inertia_:,.2f}")

fig_elbow, ax_elbow = plt.subplots(figsize=(8, 5))
ax_elbow.plot(list(k_range), inertias, marker="o", linewidth=2, color="#2563eb")
ax_elbow.set_title("Elbow Method for Optimal k Selection", fontsize=13, fontweight="bold")
ax_elbow.set_xlabel("Number of Clusters (k)")
ax_elbow.set_ylabel("Inertia")
ax_elbow.grid(True, alpha=0.3)
ax_elbow.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

# --- Programmatic elbow-point detection ------------------------------------
# Approximate the "elbow" as the k that maximizes the perpendicular
# distance from the inertia curve to the straight line connecting its
# first and last points — a standard heuristic for automating elbow
# selection without manual inspection.
k_values = np.array(list(k_range))
inertia_arr = np.array(inertias)
p1 = np.array([k_values[0], inertia_arr[0]])
p2 = np.array([k_values[-1], inertia_arr[-1]])
line_vec = p2 - p1
line_vec_norm = line_vec / np.linalg.norm(line_vec)
distances = []
for kv, inv in zip(k_values, inertia_arr):
    point = np.array([kv, inv])
    vec_to_point = point - p1
    proj_len = np.dot(vec_to_point, line_vec_norm)
    proj_point = p1 + proj_len * line_vec_norm
    distances.append(np.linalg.norm(point - proj_point))

optimal_k = int(k_values[np.argmax(distances)])
print(f"\nOptimal k selected via elbow-distance heuristic: k = {optimal_k}")
print("Justification: this k maximizes the perpendicular distance between the "
      "inertia curve and the chord connecting k=2 and k=10, i.e. the point of "
      "maximum curvature where adding clusters yields diminishing returns in "
      "within-cluster variance reduction.")

# --- Fit final KMeans model ---------------------------------------------
kmeans_final = KMeans(n_clusters=optimal_k, init="k-means++", n_init=10, random_state=42)
rfm["Segment_Number"] = kmeans_final.fit_predict(X_scaled)

# --- Segment profile table -----------------------------------------------
print(f"\n--- Segment Profile Table (k={optimal_k}) ---")
total_revenue_all = rfm["Monetary"].sum()

segment_profile = rfm.groupby("Segment_Number").agg(
    Customer_Count=("CustomerID", "count"),
    Average_Recency=("Recency", "mean"),
    Average_Frequency=("Frequency", "mean"),
    Average_Monetary=("Monetary", "mean"),
    Minimum_Monetary=("Monetary", "min"),
    Maximum_Monetary=("Monetary", "max"),
    Total_Revenue=("Monetary", "sum")
).reset_index()
segment_profile["Revenue_Share_Pct"] = (
    segment_profile["Total_Revenue"] / total_revenue_all * 100
).round(2)
segment_profile = segment_profile.round(2)
print(segment_profile.to_string(index=False))

# --- Interactive 3D Plotly scatter plot ----------------------------------
rfm["Segment_Number_Str"] = rfm["Segment_Number"].astype(str)
fig_3d = px.scatter_3d(
    rfm,
    x="Recency", y="Frequency", z="Monetary",
    color="Segment_Number_Str",
    hover_data={"CustomerID": True, "Segment_Number": True,
                "Recency": True, "Frequency": True, "Monetary": ":.2f",
                "Segment_Number_Str": False},
    title="3D Customer Segmentation: Recency vs Frequency vs Monetary",
    labels={"Segment_Number_Str": "Segment"}
)
fig_3d.update_traces(marker=dict(size=4, opacity=0.75))
fig_3d.update_layout(
    scene=dict(
        xaxis_title="Recency (days)",
        yaxis_title="Frequency (# invoices)",
        zaxis_title="Monetary (revenue)"
    ),
    legend_title_text="Segment"
)
fig_3d.show()

# ============================================================================
# SECTION 5: SEGMENT INTERPRETATION AND STRATEGIC RECOMMENDATIONS
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 5: SEGMENT INTERPRETATION AND STRATEGIC RECOMMENDATIONS")
print("=" * 90)

# --- Compute global thresholds used to interpret each segment's profile ---
overall_recency_mean = rfm["Recency"].mean()
overall_frequency_mean = rfm["Frequency"].mean()
overall_monetary_mean = rfm["Monetary"].mean()

recency_rank = segment_profile["Average_Recency"].rank(ascending=True)   # 1 = most recent
frequency_rank = segment_profile["Average_Frequency"].rank(ascending=False)  # 1 = most frequent
monetary_rank = segment_profile["Average_Monetary"].rank(ascending=False)   # 1 = highest value

def assign_label(row, n_segments):
    """
    Maps a segment's aggregate RFM profile onto a standard CRM taxonomy.
    Uses relative comparisons against the overall customer-base averages
    (rather than fixed thresholds) so the logic generalizes across
    datasets of different scale.
    """
    r, f, m = row["Average_Recency"], row["Average_Frequency"], row["Average_Monetary"]
    recent = r <= overall_recency_mean
    freq_high = f >= overall_frequency_mean
    mon_high = m >= overall_monetary_mean
    very_recent = r <= overall_recency_mean * 0.5
    very_stale = r >= overall_recency_mean * 1.5

    if recent and freq_high and mon_high and m == segment_profile["Average_Monetary"].max():
        return "Champions"
    elif recent and freq_high and mon_high:
        return "Loyal Customers"
    elif very_recent and not freq_high:
        return "Recent Customers"
    elif recent and not freq_high and not mon_high:
        return "Potential Loyalists"
    elif recent and (freq_high != mon_high):  # exactly one of the two is high
        return "Promising"
    elif recent and freq_high and not mon_high:
        return "Customers Needing Attention"
    elif not recent and not very_stale and not freq_high and not mon_high:
        return "About to Sleep"
    elif very_stale and (freq_high or mon_high):
        return "Cannot Lose Them"
    elif not recent and (freq_high or mon_high):
        return "At Risk"
    elif very_stale and not freq_high and not mon_high and m == segment_profile["Average_Monetary"].min():
        return "Lost"
    else:
        return "Hibernating"

segment_profile["Strategic_Label"] = segment_profile.apply(
    lambda row: assign_label(row, optimal_k), axis=1
)

# Map labels back onto the per-customer RFM table for export.
label_map = dict(zip(segment_profile["Segment_Number"], segment_profile["Strategic_Label"]))
rfm["Segment_Label"] = rfm["Segment_Number"].map(label_map)

# --- Final mapping table, sorted by Total Revenue descending ----------------
final_table = segment_profile[[
    "Segment_Number", "Strategic_Label", "Customer_Count",
    "Average_Recency", "Average_Frequency", "Average_Monetary",
    "Total_Revenue", "Revenue_Share_Pct"
]].sort_values("Total_Revenue", ascending=False).reset_index(drop=True)

print("\n--- Final Segment Mapping Table (sorted by Total Revenue) ---")
print(final_table.to_string(index=False))

# --- Strategic recommendations per segment ----------------------------------
recommendation_bank = {
    "Champions": "Action: Enroll in a VIP loyalty tier with early access to new "
                 "product launches and exclusive perks. Expected outcome: reinforce "
                 "brand advocacy and protect the segment generating the largest "
                 "share of revenue from competitor poaching.",
    "Loyal Customers": "Action: Launch a referral program and personalized "
                       "cross-sell bundles based on purchase history. Expected "
                       "outcome: increase average order value while leveraging "
                       "their existing trust to acquire similar high-value customers.",
    "Potential Loyalists": "Action: Send targeted onboarding offers (e.g., "
                           "second-purchase discount, membership invite) within "
                           "30 days of last purchase. Expected outcome: accelerate "
                           "conversion into the Loyal Customers segment.",
    "Recent Customers": "Action: Deploy a welcome email series introducing the "
                        "product catalog and brand story. Expected outcome: build "
                        "early engagement and establish a second-purchase habit.",
    "Promising": "Action: Use targeted cross-category promotions to grow either "
                "purchase frequency or basket size, depending on which is "
                "lagging. Expected outcome: shift the segment toward Loyal "
                "Customers within 2-3 purchase cycles.",
    "Customers Needing Attention": "Action: Trigger upsell/bundle campaigns "
                                   "highlighting premium SKUs. Expected outcome: "
                                   "raise average transaction value without "
                                   "needing to acquire new customers.",
    "About to Sleep": "Action: Send a time-limited win-back discount paired with "
                      "a preference survey. Expected outcome: re-engage before "
                      "the customer fully disengages, reducing churn rate.",
    "At Risk": "Action: Launch a personalized reactivation campaign referencing "
              "past high-value purchases with a meaningful incentive. Expected "
              "outcome: recover a portion of previously valuable customers "
              "before they churn permanently.",
    "Cannot Lose Them": "Action: Assign to a high-touch retention workflow — "
                        "personal outreach, exclusive concierge offers, or "
                        "account-manager contact. Expected outcome: prevent loss "
                        "of historically high-value customers, which would "
                        "otherwise create a disproportionate revenue gap.",
    "Hibernating": "Action: Run a low-cost automated reactivation email with a "
                  "strong discount threshold. Expected outcome: test for a "
                  "cheap revenue recovery signal before writing the segment off.",
    "Lost": "Action: Deprioritize active marketing spend; retain only in "
           "low-cost newsletter lists. Expected outcome: reallocate marketing "
           "budget toward higher-yield segments while keeping a minimal-cost "
           "reactivation channel open.",
}

print("\n--- Segment-Specific Business Recommendations ---")
for _, row in final_table.iterrows():
    label = row["Strategic_Label"]
    rec_text = recommendation_bank.get(
        label,
        "Action: Monitor engagement trends and reassess segment strategy next "
        "cycle. Expected outcome: maintain visibility into segment migration."
    )
    print(f"\nSegment {int(row['Segment_Number'])} — {label} "
          f"({int(row['Customer_Count'])} customers, "
          f"{row['Revenue_Share_Pct']}% of revenue):")
    print(f"  {rec_text}")

# ============================================================================
# SECTION 6: EXECUTIVE DASHBOARD (4-PANEL MATPLOTLIB FIGURE)
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 6: EXECUTIVE DASHBOARD")
print("=" * 90)

DARK_BG = "#0a0a0f"
plt.rcParams["text.color"] = "white"
plt.rcParams["axes.labelcolor"] = "white"
plt.rcParams["xtick.color"] = "white"
plt.rcParams["ytick.color"] = "white"

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor(DARK_BG)
for ax_row in axes:
    for ax in ax_row:
        ax.set_facecolor(DARK_BG)

palette = plt.cm.tab10(np.linspace(0, 1, len(final_table)))
segment_color_map = {
    int(seg): palette[i] for i, seg in enumerate(final_table["Segment_Number"])
}

# --- Panel 1: Horizontal bar chart of customer count per segment -----------
ax1 = axes[0, 0]
panel1_data = final_table.sort_values("Customer_Count", ascending=True)
bar_colors = [segment_color_map[int(s)] for s in panel1_data["Segment_Number"]]
bars = ax1.barh(
    panel1_data["Strategic_Label"], panel1_data["Customer_Count"], color=bar_colors
)
for bar in bars:
    width = bar.get_width()
    ax1.text(width + max(panel1_data["Customer_Count"]) * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f"{int(width):,}", va="center", fontsize=9, color="white")
ax1.set_title("Customer Count per Segment", fontsize=12, fontweight="bold", color="white")
ax1.set_xlabel("Customer Count")
ax1.grid(axis="x", alpha=0.2, color="white")

# --- Panel 2: Pie chart of revenue share per segment ------------------------
ax2 = axes[0, 1]
panel2_data = final_table.sort_values("Total_Revenue", ascending=False)
pie_colors = [segment_color_map[int(s)] for s in panel2_data["Segment_Number"]]
explode = [0.08 if i == 0 else 0 for i in range(len(panel2_data))]
ax2.pie(
    panel2_data["Revenue_Share_Pct"],
    labels=panel2_data["Strategic_Label"],
    autopct="%1.1f%%",
    colors=pie_colors,
    explode=explode,
    textprops={"color": "white", "fontsize": 8},
    pctdistance=0.75
)
ax2.set_title("Revenue Share per Segment", fontsize=12, fontweight="bold", color="white")

# --- Panel 3: Box plot of Monetary distribution per segment -----------------
# ENHANCEMENT: Added log scale (from Claude script) to better visualize
# the wide range of monetary values across segments.
ax3 = axes[1, 0]
box_data = [rfm.loc[rfm["Segment_Number"] == seg, "Monetary"].values
            for seg in final_table["Segment_Number"]]
box_labels = final_table["Strategic_Label"].tolist()
bp = ax3.boxplot(box_data, labels=box_labels, patch_artist=True,
                  flierprops=dict(marker="o", markersize=3, markerfacecolor="white",
                                  markeredgecolor="white", alpha=0.5))
for patch, seg in zip(bp["boxes"], final_table["Segment_Number"]):
    patch.set_facecolor(segment_color_map[int(seg)])
    patch.set_alpha(0.7)
for element in ["whiskers", "caps", "medians"]:
    for line in bp[element]:
        line.set_color("white")
ax3.set_yscale("log")  # <-- ADDED: Log scale improves visibility across orders of magnitude
ax3.set_title("Monetary Value Distribution per Segment (Log Scale)", fontsize=12, fontweight="bold", color="white")
ax3.set_ylabel("Monetary (revenue, log scale)")
ax3.tick_params(axis="x", rotation=45)
ax3.grid(axis="y", alpha=0.2, color="white")

# --- Panel 4: Scatter plot of Recency vs Frequency, colored by segment ------
ax4 = axes[1, 1]
for seg in final_table["Segment_Number"]:
    seg_data = rfm[rfm["Segment_Number"] == seg]
    ax4.scatter(
        seg_data["Recency"], seg_data["Frequency"],
        s=18, alpha=0.6, color=segment_color_map[int(seg)],
        label=label_map[seg]
    )
ax4.set_title("Recency vs Frequency by Segment", fontsize=12, fontweight="bold", color="white")
ax4.set_xlabel("Recency (days)")
ax4.set_ylabel("Frequency (# invoices)")
ax4.grid(alpha=0.2, color="white")
ax4.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8,
           facecolor=DARK_BG, edgecolor="white", labelcolor="white")

fig.suptitle("Customer Segmentation Analysis — RFM Model Results",
             fontsize=16, fontweight="bold", color="white")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# ============================================================================
# SECTION 7: ANALYTICAL EXPORTS FOR BUSINESS INTELLIGENCE
# ============================================================================
print("\n" + "=" * 90)
print("SECTION 7: ANALYTICAL EXPORTS FOR BUSINESS INTELLIGENCE")
print("=" * 90)

customer_export = rfm[[
    "CustomerID", "Recency", "Frequency", "Monetary",
    "Segment_Number", "Segment_Label"
]].copy()
customer_export.to_csv("customer_segments.csv", index=False)

segment_export = final_table.rename(columns={
    "Strategic_Label": "Segment_Label",
    "Customer_Count": "Customer_Count",
    "Average_Recency": "Avg_Recency",
    "Average_Frequency": "Avg_Frequency",
    "Average_Monetary": "Avg_Monetary",
    "Total_Revenue": "Total_Revenue",
    "Revenue_Share_Pct": "Revenue_Share_Pct"
})[[
    "Segment_Number", "Segment_Label", "Customer_Count", "Avg_Recency",
    "Avg_Frequency", "Avg_Monetary", "Total_Revenue", "Revenue_Share_Pct"
]]
segment_export.to_csv("segment_summary.csv", index=False)

print(f"\nExported customer_segments.csv ({len(customer_export)} rows) and "
      f"segment_summary.csv ({len(segment_export)} rows) for Power BI and "
      f"Tableau dashboard integration.")

print("\n" + "=" * 90)
print("PIPELINE COMPLETE")
print("=" * 90)

CUSTOMER SEGMENTATION PIPELINE — RFM ANALYSIS + KMEANS CLUSTERING

SECTION 1: DATA LOADING AND VALIDATION

Loading dataset from UCI ML Repository:
  https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx
